# NB 3A — Porkchop Plot Generator: Earth–Mars Launch Window Analysis

## Executive Summary
A porkchop plot is an engineering map that shows how mission cost changes with departure date and transfer duration. In this notebook, you will compute Earth–Mars Lambert transfers over a grid of launch dates and times of flight, then visualize launch energy and arrival conditions to identify practical launch windows.

## Learning Objectives
- Understand what a launch window is and why it exists.
- Understand how Lambert transfers are used for interplanetary mission design.
- Compute launch characteristic energy and arrival hyperbolic excess speed.
- Interpret why porkchop contours have their characteristic shape.
- Relate trajectory design outputs to launch vehicle capability.

## Theoretical Background
Lambert's problem maps boundary positions and time of flight to transfer velocities:

$$
\mathbf{r}_1, \mathbf{r}_2, \Delta t \rightarrow \mathbf{v}_1, \mathbf{v}_2
$$

Departure hyperbolic excess velocity:

$$
\mathbf{v}_{\infty,dep} = \mathbf{v}_{sc,dep} - \mathbf{v}_{Earth}
$$

Arrival hyperbolic excess velocity:

$$
\mathbf{v}_{\infty,arr} = \mathbf{v}_{sc,arr} - \mathbf{v}_{Mars}
$$

Launch characteristic energy:

$$
C_3 = \left\| \mathbf{v}_{\infty,dep} \right\|^2
$$

Approximate heliocentric matching cost:

$$
\Delta v_{total} \approx \left\| \mathbf{v}_{\infty,dep} \right\| + \left\| \mathbf{v}_{\infty,arr} \right\|
$$

- $$C_3$$ is not full launch vehicle ascent $$\Delta v$$.
- $$C_3$$ measures excess energy after Earth escape.
- Lower $$C_3$$ generally means easier launch requirements.
- Lower arrival $$v_\infty$$ generally means easier capture at Mars.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from scipy import constants
from astropy import units as u
from astropy.time import Time
from astropy.coordinates import solar_system_ephemeris

MU_SUN_KM3_S2 = 1.32712440018e11
KM_PER_AU = 149597870.700
SECONDS_PER_DAY = 86400.0

print("Units used: km, km/s, seconds, days; AU only as optional reference.")

In [ ]:
HAS_ASTROQUERY = False
HAS_POLIASTRO = False

try:
    from astroquery.jplhorizons import Horizons
    HAS_ASTROQUERY = True
except Exception:
    pass

try:
    from poliastro.iod import izzo as iod_izzo
    HAS_POLIASTRO = True
except Exception:
    pass

print(f"astroquery available: {HAS_ASTROQUERY}")
print(f"poliastro available: {HAS_POLIASTRO}")

In [ ]:
def get_body_state(body_name, epoch):
    body_key = body_name.strip().lower()
    if body_key not in ["earth", "mars"]:
        raise ValueError("body_name must be 'earth' or 'mars'")

    if HAS_ASTROQUERY:
        try:
            target_id = "399" if body_key == "earth" else "499"
            obj = Horizons(id=target_id, id_type='majorbody', location='@sun', epochs=epoch.tdb.jd)
            vec = obj.vectors()
            r = np.array([vec['x'][0], vec['y'][0], vec['z'][0]], dtype=float) * KM_PER_AU
            v = np.array([vec['vx'][0], vec['vy'][0], vec['vz'][0]], dtype=float) * KM_PER_AU / SECONDS_PER_DAY
            return r, v, 'heliocentric inertial (approx ecliptic)', 'JPL Horizons (astroquery)'
        except Exception:
            pass

    from astropy.coordinates import get_body_barycentric_posvel
    with solar_system_ephemeris.set('builtin'):
        pos, vel = get_body_barycentric_posvel(body_key, epoch)
    return pos.xyz.to(u.km).value, vel.xyz.to(u.km/u.s).value, 'barycentric inertial', 'Astropy builtin ephemeris fallback' 

In [ ]:
def solve_lambert_transfer(r1, r2, tof_seconds, mu_sun=MU_SUN_KM3_S2):
    if not HAS_POLIASTRO:
        raise ImportError("poliastro Lambert solver unavailable")
    k = mu_sun * (u.km**3 / u.s**2)
    sols = iod_izzo.lambert(k, r1*u.km, r2*u.km, tof_seconds*u.s)
    try:
        v1q, v2q = next(iter(sols))
    except TypeError:
        v1q, v2q = sols
    return v1q.to(u.km/u.s).value, v2q.to(u.km/u.s).value

In [ ]:
start_date = datetime(2026,1,1)
end_date = datetime(2027,12,31)
dep_step_days = 5
tof_days = np.arange(100,301,5)

dep_dates=[]
d=start_date
while d<=end_date:
    dep_dates.append(d)
    d += timedelta(days=dep_step_days)
dep_dates=np.array(dep_dates)

nt, nd = len(tof_days), len(dep_dates)
C3_grid = np.full((nt, nd), np.nan)
vinf_dep_grid = np.full((nt, nd), np.nan)
vinf_arr_grid = np.full((nt, nd), np.nan)
total_vinf_grid = np.full((nt, nd), np.nan)
arrival_dates_grid = np.empty((nt, nd), dtype=object)

print(f"Grid points: {nt*nd}")

In [ ]:
fail_count=0
ok_count=0
used_methods=set()

for j, dep_dt in enumerate(dep_dates):
    dep_epoch = Time(dep_dt, scale='tdb')
    try:
        rE,vE,_,mdep = get_body_state('earth', dep_epoch)
        used_methods.add(mdep)
    except Exception:
        fail_count += len(tof_days)
        continue

    for i,tof_d in enumerate(tof_days):
        arr_dt = dep_dt + timedelta(days=int(tof_d))
        arrival_dates_grid[i,j] = arr_dt
        try:
            rM,vM,_,marr = get_body_state('mars', Time(arr_dt, scale='tdb'))
            used_methods.add(marr)
            v1,v2 = solve_lambert_transfer(rE,rM,float(tof_d)*SECONDS_PER_DAY,MU_SUN_KM3_S2)
            vinf_dep = np.linalg.norm(v1-vE)
            vinf_arr = np.linalg.norm(v2-vM)
            C3_grid[i,j] = vinf_dep**2
            vinf_dep_grid[i,j] = vinf_dep
            vinf_arr_grid[i,j] = vinf_arr
            total_vinf_grid[i,j] = vinf_dep + vinf_arr
            ok_count += 1
        except Exception:
            fail_count += 1

print("Method(s) used:")
for m in sorted(used_methods):
    print("-", m)
print("Success:", ok_count, " Fail:", fail_count)

In [ ]:
valid = np.isfinite(C3_grid)
print("Any valid Lambert solutions:", np.any(valid))
if np.any(valid):
    print("C3 non-negative minimum:", np.nanmin(C3_grid) >= 0.0)
    print("Minimum C3 finite:", np.isfinite(np.nanmin(C3_grid)))
ratio = fail_count/(fail_count+ok_count) if (fail_count+ok_count)>0 else 1.0
if ratio > 0.4:
    print("WARNING: More than 40% of grid points failed.")

In [ ]:
idx=np.nanargmin(C3_grid)
i_best,j_best=np.unravel_index(idx,C3_grid.shape)
best_dep_date=dep_dates[j_best]
best_arr_date=arrival_dates_grid[i_best,j_best]
best_tof=tof_days[i_best]
best_C3=C3_grid[i_best,j_best]
best_vinf_dep=vinf_dep_grid[i_best,j_best]
best_vinf_arr=vinf_arr_grid[i_best,j_best]
best_total_vinf=total_vinf_grid[i_best,j_best]

print("Best departure date:", best_dep_date.date())
print("Best arrival date:", best_arr_date.date())
print("Best TOF [days]:", best_tof)
print("Minimum C3 [km^2/s^2]:", round(best_C3,3))
print("Departure v_inf [km/s]:", round(best_vinf_dep,3))
print("Arrival v_inf [km/s]:", round(best_vinf_arr,3))
print("Total approx v_inf [km/s]:", round(best_total_vinf,3))

In [ ]:
X,Y=np.meshgrid(dep_dates,tof_days)
fig,ax=plt.subplots(figsize=(13,7))
cf=ax.contourf(X,Y,C3_grid,levels=20,cmap='viridis')
cb=plt.colorbar(cf,ax=ax)
cb.set_label('C3 [km^2/s^2]')

if np.any(np.isfinite(vinf_arr_grid)):
    vmin=np.nanpercentile(vinf_arr_grid,15)
    vmax=np.nanpercentile(vinf_arr_grid,85)
    if vmax>vmin:
        cs=ax.contour(X,Y,vinf_arr_grid,levels=np.linspace(vmin,vmax,8),colors='white',linestyles='dashed',linewidths=1)
        ax.clabel(cs,inline=True,fontsize=8,fmt='%.1f km/s')

constraint=np.where(C3_grid>20.0,1.0,np.nan)
ax.contourf(X,Y,constraint,levels=[0.5,1.5],colors=['red'],alpha=0.18)
ax.text(dep_dates[int(0.05*len(dep_dates))],tof_days[int(0.85*len(tof_days))],'Infeasible
(C3 > 20)',color='red')
ax.text(dep_dates[int(0.60*len(dep_dates))],tof_days[int(0.20*len(tof_days))],'Feasible
(C3 ≤ 20)',color='white')

ax.set_title('Earth–Mars Porkchop Plot: C3 with Arrival v_inf Contours')
ax.set_xlabel('Departure Date')
ax.set_ylabel('Time of Flight [days]')
ax.grid(True,alpha=0.3)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.get_xticklabels(),rotation=30,ha='right')
plt.tight_layout(); plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(13,6))
ct=ax.contourf(X,Y,total_vinf_grid,levels=20,cmap='plasma')
cb=plt.colorbar(ct,ax=ax)
cb.set_label('Approx total v_inf [km/s]')
ax.set_title('Earth–Mars Transfer Approximate Matching Cost')
ax.set_xlabel('Departure Date'); ax.set_ylabel('Time of Flight [days]')
ax.grid(True,alpha=0.3)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.get_xticklabels(),rotation=30,ha='right')
plt.tight_layout(); plt.show()

In [ ]:
threshold=best_C3+5.0
mask=(C3_grid<=threshold)&np.isfinite(C3_grid)
if np.any(mask):
    js=np.unique(np.where(mask)[1])
    start=dep_dates[js].min(); end=dep_dates[js].max()
    print("Threshold C3 <= C3_min + 5:", round(threshold,3))
    print("Window start:", start.date())
    print("Window end:", end.date())
    print("Window width [days]:", (end-start).days)
else:
    print("No practical window points found")

In [ ]:
late_date=best_dep_date+timedelta(days=14)
j_late=int(np.argmin([abs((d-late_date).days) for d in dep_dates]))
i_late=i_best
if not np.isfinite(C3_grid[i_late,j_late]) and np.any(np.isfinite(C3_grid[:,j_late])):
    i_late=int(np.nanargmin(C3_grid[:,j_late]))

late_C3=C3_grid[i_late,j_late]
late_vinf_dep=vinf_dep_grid[i_late,j_late]
late_vinf_arr=vinf_arr_grid[i_late,j_late]

print("C3 at optimum:", round(best_C3,3))
print("C3 after ~2-week delay:", round(late_C3,3))
print("C3 penalty [km^2/s^2]:", round(late_C3-best_C3,3))
print("Opt dep/arr v_inf [km/s]:", round(best_vinf_dep,3), round(best_vinf_arr,3))
print("Late dep/arr v_inf [km/s]:", round(late_vinf_dep,3), round(late_vinf_arr,3))
print("Interpretation: delayed launch usually raises energy and/or arrival speed demand.")

## Discussion
- The practical window width comes from dates where $$C_3 \le C_{3,min}+5$$.
- The two-week delay penalty quantifies schedule sensitivity near the optimum.
- The porkchop shape comes from orbital phasing and transfer-time coupling.
- Both $$C_3$$ and arrival $$v_\infty$$ are needed because launch and capture constraints differ.
- Launch vehicle capability shapes feasibility by excluding high-$$C_3$$ regions.

## Industrial Interpretation
Mission designers use porkchop plots during early mission feasibility trades. The metric $$C_3$$ links transfer design to launch vehicle selection, and arrival $$v_\infty$$ links transfer design to capture propulsion and entry strategies. This is a first-order design tool used before higher-fidelity optimization and operational refinement.

## Engineering Takeaways
- Lambert-based grids map interplanetary opportunities in a compact visual form.
- Minimum $$C_3$$ points identify energetically favorable launch opportunities.
- Arrival $$v_\infty$$ matters for capture cost and mission architecture.
- Feasibility depends on both trajectory physics and launcher performance bounds.
- Even short schedule delays can create measurable mission penalties.